In [1]:
import json
import re
import time

import pandas as pd
import requests
from bs4 import BeautifulSoup

In [ ]:
with open("Datos/diputados.json", encoding="utf-8") as f:
    data = json.load(f) #abre el json de diputados y crea un diccionario de python.

ULTIMO_ID = max(int(v["id"]) for v in data["votaciones"]) #para cada votacion toma su id y se queda con los mayores
print(ULTIMO_ID) #numero maximo

5902


In [ ]:
BASE = "https://votaciones.hcdn.gob.ar" #direccion del sitio

sesion = requests.Session() #una pestaña de navegador que se reutiliza 
sesion.headers["User-Agent"] = "Mozilla/5.0 (proyecto academico UBA)"#se presenta el agente, le dice quien pide las paginas

inicio = sesion.get(BASE, timeout=60) #descarga la pagina principal
todos = sorted({int(n) for n in re.findall(r"votacion/(\d+)\.html", inicio.text)}) #busca votaciones html y lass ordena
nuevos = [n for n in todos if n > ULTIMO_ID] #busca los que sean mayores al ultimo id

print("Votaciones en el sitio:", len(todos), "| Nuevas:", len(nuevos)) #trae las votaciones y las nuevas

Votaciones en el sitio: 2319 | Nuevas: 79


In [ ]:
def leer_votacion(numero): #crea la funcion leer votacion 
    respuesta = sesion.get(f"{BASE}/votacion/{numero}.html", timeout=30) #descarga la pagina de esa votacion
    if respuesta.status_code != 200: #si la respuesta es distinta a 200 devuelve lista vacia
        return [] 
    respuesta.encoding = "utf-8" #corrige las tildes 
    pagina = BeautifulSoup(respuesta.text, "html.parser") #convierte el html en algo navegable 

    encabezado = pagina.find("h4").get_text(" ", strip=True) #busca h4 que trae el titulo y la fecha
    fecha = re.search(r"\d{2}/\d{2}/\d{4}\s*-\s*\d{2}:\d{2}", encabezado) #Busca la fecha dentro del encabezado. El patrón es: 2 dígitos / 2 dígitos / 4 dígitos, un guion y la hora.
    titulo = encabezado[:fecha.start()].strip() if fecha else encabezado #El título es lo que está antes de la fecha. [:fecha.start()] corta el texto hasta donde empieza la fecha. Si no hubo fecha, usa el encabezado entero.

    resultado = pagina.find("h3").get_text(strip=True) #resultado afirmativo o negativo esta en el h3

    filas = [] #lista vacia donde se van a ir juntando las filas
    for fila in pagina.find("table").find_all("tr"): #recorre todas las filas de la tabla 
        celdas = [td.get_text(strip=True) for td in fila.find_all("td")] #consigue los textos de la celda de esa fila
        if len(celdas) < 5: #salta las celdas que no son de diputados
            continue
        filas.append({
            "votacion_id": numero,
            "titulo": titulo,
            "fecha": fecha.group(0) if fecha else "",
            "resultado_general": resultado,
            "diputado": celdas[1],
            "bloque": celdas[2],
            "provincia": celdas[3],
            "voto": celdas[4],
        }) #Agrega una fila con los datos. Un diccionario con las mismas columnas que votos_expandidos.csv.
    return filas

In [ ]:
prueba = leer_votacion(nuevos[-1]) #lee la ultim votacion nueva

print(len(prueba))
prueba[0]

257


{'votacion_id': 6026,
 'titulo': 'MOCIÓN SOLICITADA POR LA DIP. GIUDICI, SILVANA.',
 'fecha': '22/10/2026 - 03:28',
 'resultado_general': 'AFIRMATIVO',
 'diputado': 'YEDLIN, PABLO RAUL',
 'bloque': 'Union Por La Patria',
 'provincia': 'Tucumán',
 'voto': 'NEGATIVO'}

In [ ]:
todas_las_filas = [] #lista vacia para adjuntar las filas de todas las votaciones 

for i, numero in enumerate(nuevos, start=1): #recorre las votaciones nuevas y las enumera.
    todas_las_filas.extend(leer_votacion(numero)) #lee una votacion  y suma sus filas 
    time.sleep(1) #espera un segundo para el pedido de la siguiente
    if i % 20 == 0: #muestra el avance cada 20
        print(i, "de", len(nuevos))

print("Filas descargadas:", len(todas_las_filas))

20 de 79


40 de 79


60 de 79


Filas descargadas: 20303


In [ ]:
nuevos_df = pd.DataFrame(todas_las_filas) #convierta la llista en tabla de pandas

nuevos_df.to_csv("Datos/votos_nuevos.csv", index=False, encoding="utf-8-sig") #guarda el csv

print(nuevos_df["votacion_id"].nunique(), "votaciones") #cuantas votaciones distintas 
print(nuevos_df["fecha"].iloc[0], "a", nuevos_df["fecha"].iloc[-1]) #primera y ultima fecha
nuevos_df["voto"].value_counts() #cuantos votos de cada tipo

79 votaciones
20/05/2026 - 11:46 a 22/10/2026 - 03:28


voto
AFIRMATIVO    13578
NEGATIVO       4869
AUSENTE        1596
ABSTENCION      181
PRESIDENTE       79
Name: count, dtype: int64